# OOP 4: Special Methods (Dunder Methods)

## Table of Contents

1. [What are Special Methods?](#1)
2. [__init__: Object Initialization](#2)
3. [__str__ and __repr__: Object Representation](#3)
4. [__len__, __getitem__, __contains__: Collection Protocol](#4)
5. [__iter__ and __next__: Making Objects Iterable](#5)
6. [__enter__ and __exit__: Context Managers](#6)
7. [__bool__: Truth Testing](#7)
8. [__call__: Callable Objects](#8)
9. [__eq__, __lt__ and Friends: Comparison and Ordering](#9)
10. [__hash__: The Consequence of __eq__](#10)
11. [__add__ and Arithmetic Methods](#11)
12. [Exercise: Extend the Dataset Class](#12)

---
## 1. What are Special Methods? <a id="1"></a>

Special methods (also called **dunder methods**, for "double underscore") let Python's built-in syntax work with your objects.

```python
len(dataset)         # calls dataset.__len__()
dataset[0]           # calls dataset.__getitem__(0)
print(dataset)       # calls dataset.__str__()
with dataset: ...    # calls dataset.__enter__() and __exit__()
```

You implement these methods, Python calls them. This is called the **data model** and it's how sklearn's `.fit()/.transform()`, pandas DataFrames, and everything else in Python's ecosystem is built.

---
## 2. __init__: Object Initialization <a id="2"></a>

Already seen in notebook 01, but worth formalising: `__init__` is called automatically when you create an object. It receives any arguments you pass to the class.

The convention for `__init__` is: *set up the object's attributes and nothing else*. Don't do expensive work (file I/O, network calls) in `__init__`. Use factory methods or lazy loading instead.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


class Dataset:
    def __init__(self, X: pd.DataFrame, y: pd.Series, name: str = "dataset"):
        # __init__: just set attributes, no side effects
        self.X = X
        self.y = y
        self.name = name
        self._splits: dict = {}

    def split(self, test_size: float = 0.2, random_state: int = 42) -> "Dataset":
        X_tr, X_te, y_tr, y_te = train_test_split(
            self.X, self.y, test_size=test_size,
            random_state=random_state, stratify=self.y
        )
        self._splits = {"X_train": X_tr, "X_test": X_te,
                        "y_train": y_tr, "y_test": y_te}
        return self

    @property
    def train(self): return self._splits["X_train"], self._splits["y_train"]
    @property
    def test(self):  return self._splits["X_test"],  self._splits["y_test"]


np.random.seed(42)
X = pd.DataFrame({"tenure": np.random.randint(1, 60, 100),
                  "charges": np.random.uniform(20, 100, 100)})
y = pd.Series((np.random.rand(100) > 0.7).astype(int), name="churn")

ds = Dataset(X, y, "churn").split()
print(f"Dataset created: {ds.name}, {ds.n_samples if hasattr(ds, 'n_samples') else len(ds.X)} rows")

---
## 3. __str__ and __repr__: Object Representation <a id="3"></a>

- `__str__`: for easy reading (called by `print()` and `str()`)
- `__repr__`: for developers (called in the REPL and by `repr()`)

Rule of thumb: `__repr__` should show enough to reconstruct the object (or at least identify it). `__str__` should be readable.

In [ ]:
class Dataset:
    def __init__(self, X: pd.DataFrame, y: pd.Series, name: str = "dataset"):
        self.X = X
        self.y = y
        self.name = name
        self._splits: dict = {}

    def split(self, test_size: float = 0.2, random_state: int = 42) -> "Dataset":
        X_tr, X_te, y_tr, y_te = train_test_split(
            self.X, self.y, test_size=test_size, random_state=random_state, stratify=self.y
        )
        self._splits = {"X_train": X_tr, "X_test": X_te, "y_train": y_tr, "y_test": y_te}
        return self

    @property
    def train(self): return self._splits["X_train"], self._splits["y_train"]
    @property
    def test(self):  return self._splits["X_test"],  self._splits["y_test"]

    # --- string representations ---

    def __str__(self) -> str:
        n, p = self.X.shape
        pos = int(self.y.sum())
        split_info = "split" if self._splits else "not split"
        return (
            f"Dataset('{self.name}'): {n} rows x {p} features, "
            f"{pos}/{n} positive ({split_info})"
        )

    def __repr__(self) -> str:
        return (
            f"Dataset(name={self.name!r}, "
            f"shape={self.X.shape}, "
            f"positive_rate={self.y.mean():.2f})"
        )


ds = Dataset(X, y, "churn").split()

print(str(ds))   # __str__
print(repr(ds))  # __repr__
print()
ds               # Jupyter/REPL uses __repr__

---
## 4. __len__, __getitem__, __setitem__, __contains__: Collection Protocol <a id="4"></a>

These methods let your objects behave like containers (lists, dicts). Adding them to `Dataset` makes it intuitive to work with:

```python
len(ds)    # how many rows?
ds[0]      # first row?
ds["tenure"]  # a column?
```

In [ ]:
class Dataset:
    def __init__(self, X: pd.DataFrame, y: pd.Series, name: str = "dataset"):
        self.X = X
        self.y = y
        self.name = name
        self._splits: dict = {}

    def split(self, test_size: float = 0.2, random_state: int = 42) -> "Dataset":
        X_tr, X_te, y_tr, y_te = train_test_split(
            self.X, self.y, test_size=test_size, random_state=random_state, stratify=self.y
        )
        self._splits = {"X_train": X_tr, "X_test": X_te, "y_train": y_tr, "y_test": y_te}
        return self

    @property
    def train(self): return self._splits["X_train"], self._splits["y_train"]
    @property
    def test(self):  return self._splits["X_test"],  self._splits["y_test"]
    @property
    def feature_names(self): return self.X.columns.tolist()

    def __str__(self) -> str:
        n, p = self.X.shape
        pos = int(self.y.sum())
        return f"Dataset('{self.name}'): {n} rows x {p} features, {pos}/{n} positive"

    def __repr__(self) -> str:
        return f"Dataset(name={self.name!r}, shape={self.X.shape})"

    # --- collection protocol ---

    def __len__(self) -> int:
        """len(ds) returns number of rows."""
        return len(self.X)

    def __getitem__(self, key):
        """ds[col] returns a column. ds[i] returns row i."""
        if isinstance(key, str):
            return self.X[key]
        return self.X.iloc[key]

    def __contains__(self, col: str) -> bool:
        """'tenure' in ds — checks if column exists."""
        return col in self.X.columns


ds = Dataset(X, y, "churn")

print("len(ds):", len(ds))
print("ds['tenure']:", ds["tenure"][:5].tolist())
print("ds[0]:"); print(ds[0])
print("'tenure' in ds:", "tenure" in ds)
print("'foo' in ds:   ", "foo" in ds)

---
## 5. __iter__ and __next__: Making Objects Iterable <a id="5"></a>

`__iter__` is called when Python needs an iterator from your object, like in a `for` loop, `list()`, `zip()`, etc.
It should return an object with a `__next__` method.

The simplest way: use `yield` inside `__iter__`, which makes it a **generator** and handles `__next__` automatically.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

class Dataset:
    def __init__(self, X: pd.DataFrame, y: pd.Series, name: str = "dataset"):
        self.X = X
        self.y = y
        self.name = name
        self._splits: dict = {}

    def split(self, test_size: float = 0.2, random_state: int = 42) -> "Dataset":
        X_tr, X_te, y_tr, y_te = train_test_split(
            self.X, self.y, test_size=test_size,
            random_state=random_state, stratify=self.y
        )
        self._splits = {"X_train": X_tr, "X_test": X_te,
                        "y_train": y_tr, "y_test": y_te}
        return self

    @property
    def train(self): return self._splits["X_train"], self._splits["y_train"]
    @property
    def test(self):  return self._splits["X_test"],  self._splits["y_test"]
    @property
    def feature_names(self) -> list[str]: return self.X.columns.tolist()

    def __len__(self) -> int: return len(self.X)
    def __getitem__(self, key): return self.X[key] if isinstance(key, str) else self.X.iloc[key]
    def __contains__(self, col: str) -> bool: return col in self.X.columns
    def __str__(self) -> str:
        n, p = self.X.shape
        return f"Dataset('{self.name}'): {n} rows x {p} features"
    def __repr__(self) -> str:
        return f"Dataset(name={self.name!r}, shape={self.X.shape})"

    def __iter__(self):
        """Iterate over rows of X as Series objects."""
        for _, row in self.X.iterrows():
            yield row


np.random.seed(42)
X = pd.DataFrame({"tenure": np.random.randint(1, 60, 5),
                  "charges": np.random.uniform(20, 100, 5)})
y = pd.Series([1, 0, 1, 0, 1], name="churn")
ds = Dataset(X, y, "demo")

# for loop uses __iter__
for row in ds:
    print(row.to_dict())

In [ ]:
# __iter__ also powers: list(), zip(), unpacking, comprehensions
rows = list(ds)
print(f"list(ds) gives {len(rows)} Series objects")

# zip two datasets together
ds2 = Dataset(X.copy(), y.copy(), "demo2")
for row_a, row_b in zip(ds, ds2):
    pass  # iterates both in lockstep
print("zip(ds, ds2) works")

# comprehension
tenures = [row["tenure"] for row in ds]
print("Tenures via comprehension:", tenures)

---
## 6. __enter__ and __exit__: Context Managers <a id="6"></a>

Context managers are the `with statement`. They guarantee cleanup code runs even if an exception is raised. A perfect solution for database connections, file handles, temporary state.

You've used them constantly: `with open(...) as f`, `with pd.option_context(...)`. Here's how they work internally.

In [ ]:
import sqlite3
import tempfile
import os


class DatabaseConnection:
    """A database connection that opens on entry and closes on exit.

    Usage:
        with DatabaseConnection("data.db") as conn:
            conn.execute("SELECT ...")
    # Connection is closed here, even if an exception occurred.
    """

    def __init__(self, db_path: str):
        self.db_path = db_path
        self._conn = None

    def __enter__(self) -> "DatabaseConnection":
        """Called at the start of the with block."""
        self._conn = sqlite3.connect(self.db_path)
        print(f"Connected to {self.db_path}")
        return self   # the value assigned to the `as` variable

    def __exit__(self, exc_type, exc_val, exc_tb) -> bool:
        """Called at the end of the with block (even on exception)."""
        if self._conn:
            self._conn.close()
            print(f"Connection to {self.db_path} closed")
        # Return False (or None) to let exceptions propagate
        return False

    def execute(self, sql: str, params=()):
        return self._conn.execute(sql, params)


# Create a temp DB to demo
with tempfile.NamedTemporaryFile(suffix=".db", delete=False) as f:
    tmp_db = f.name

try:
    with DatabaseConnection(tmp_db) as db:
        db.execute("CREATE TABLE IF NOT EXISTS runs (id INTEGER PRIMARY KEY, score REAL)")
        db.execute("INSERT INTO runs (score) VALUES (?)", (0.87,))
        db.execute("INSERT INTO runs (score) VALUES (?)", (0.91,))
        rows = list(db.execute("SELECT * FROM runs"))
        print("Rows:", rows)
    # Connection is definitely closed here
finally:
    os.unlink(tmp_db)

In [ ]:
# A second use case: temporarily suppressing sklearn's ConvergenceWarning
import warnings
from contextlib import contextmanager

@contextmanager
def suppress_warnings(*categories):
    """Temporarily suppress specific warning categories."""
    with warnings.catch_warnings():
        for cat in categories:
            warnings.simplefilter("ignore", cat)
        yield


from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression

# Without suppression
print("Without suppression:")
try:
    _ = LogisticRegression(max_iter=1).fit(X, y)
except Exception as e:
    print(f"  Error: {e}")

In [ ]:
# With suppression
print("With suppression:")
with suppress_warnings(ConvergenceWarning):
    lr = LogisticRegression(max_iter=1).fit(X, y)
    print(f"  Fitted without warning noise")

---
## 7. __bool__: Truth Testing <a id="7"></a>

`__bool__` is called whenever Python needs to evaluate your object as `True` or `False` in an `if` statement, `while`, `assert`, or boolean expressions.

Without `__bool__`, Python falls back to `__len__` (non-zero = True). If neither exists, the object is always `True`.

For ML objects, `__bool__` gives you expressive guards:
```python
if not scaler:
    raise RuntimeError("Scaler is not fitted yet.")
```

In [ ]:
class StandardScaler:
    """Scales features to zero mean and unit variance."""

    def __init__(self):
        self._mean = None
        self._std = None

    def fit(self, X: pd.DataFrame) -> "StandardScaler":
        self._mean = X.mean()
        self._std = X.std().clip(lower=1e-8)
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if not self:  # uses __bool__
            raise RuntimeError("Scaler is not fitted. Call fit() first.")
        return (X - self._mean) / self._std

    def __bool__(self) -> bool:
        """True if the scaler has been fitted."""
        return self._mean is not None

    def __repr__(self) -> str:
        return f"StandardScaler(fitted={bool(self)})"


scaler = StandardScaler()
print(f"Before fit: bool(scaler) = {bool(scaler)}")
print(f"  if scaler: ... → {'runs' if scaler else 'skipped'}")

X = pd.DataFrame({"tenure": [12.0, 5.0, 24.0], "charges": [70.0, 30.0, 90.0]})
scaler.fit(X)
print(f"After  fit: bool(scaler) = {bool(scaler)}")
print(f"  if scaler: ... → {'runs' if scaler else 'skipped'}")

# Guard pattern
if not scaler:
    print("Would raise here")
else:
    print("Safe to transform:", scaler.transform(X).round(2).to_dict())

---
## 8. __call__: Callable Objects <a id="8"></a>

`__call__` makes an instance behave like a function. You invoke it with `()`.

This is extremely common in ML:
- **sklearn transformers** use `fit_transform()` but PyTorch modules use `model(X)`, that's `__call__`
- **Custom metrics**: pass an object to an evaluation loop instead of a plain function
- **Stateful transforms**: a callable that remembers parameters between calls

In [ ]:
class ThresholdClassifier:
    """Classifies based on a single feature threshold.

    Once fitted, can be called directly: classifier(X)
    """

    def __init__(self, feature: str, threshold: float | None = None):
        self.feature = feature
        self.threshold = threshold

    def fit(self, X: pd.DataFrame, y: pd.Series) -> "ThresholdClassifier":
        """Find the threshold that maximises accuracy on training data."""
        best_acc, best_thresh = 0.0, 0.0
        for thresh in sorted(X[self.feature].unique()):
            preds = (X[self.feature] >= thresh).astype(int)
            acc = (preds == y).mean()
            if acc > best_acc:
                best_acc, best_thresh = acc, thresh
        self.threshold = best_thresh
        return self

    def __call__(self, X: pd.DataFrame) -> np.ndarray:
        """Classify X. Allows: preds = classifier(X)"""
        if self.threshold is None:
            raise RuntimeError("Call fit() before predicting.")
        return (X[self.feature] >= self.threshold).astype(int).values

    def __repr__(self) -> str:
        return f"ThresholdClassifier(feature={self.feature!r}, threshold={self.threshold})"


np.random.seed(42)
X_tr = pd.DataFrame({"tenure": np.random.randint(1, 60, 100)})
y_tr = pd.Series((X_tr["tenure"] < 15).astype(int))
X_te = pd.DataFrame({"tenure": np.random.randint(1, 60, 20)})

clf = ThresholdClassifier(feature="tenure")
clf.fit(X_tr, y_tr)
print(clf)

# Call it like a function
preds = clf(X_te)
print("Predictions:", preds)

In [ ]:
# __call__ also enables objects to be passed where functions are expected

from sklearn.metrics import accuracy_score

class WeightedAccuracy:
    """Accuracy metric that penalises false negatives more heavily."""

    def __init__(self, fn_weight: float = 2.0):
        self.fn_weight = fn_weight

    def __call__(self, y_true: pd.Series, y_pred: np.ndarray) -> float:
        """Compute weighted accuracy. Callable like a plain function."""
        tp = ((y_pred == 1) & (y_true == 1)).sum()
        tn = ((y_pred == 0) & (y_true == 0)).sum()
        fp = ((y_pred == 1) & (y_true == 0)).sum()
        fn = ((y_pred == 0) & (y_true == 1)).sum()
        return (tp + tn) / (tp + tn + fp + self.fn_weight * fn)

    def __repr__(self) -> str:
        return f"WeightedAccuracy(fn_weight={self.fn_weight})"


y_true = pd.Series([1, 0, 1, 1, 0, 1, 0, 0])
y_pred = np.array(    [1, 0, 0, 1, 0, 1, 1, 0])

standard  = accuracy_score
weighted  = WeightedAccuracy(fn_weight=3.0)

print(f"Standard accuracy:  {standard(y_true, y_pred):.3f}")
print(f"Weighted accuracy:  {weighted(y_true, y_pred):.3f}")
print()

# Pass either to a higher-order function — both are just callables
def evaluate(metric, y_true, y_pred):
    return metric(y_true, y_pred)

print("evaluate() with standard:", round(evaluate(standard, y_true, y_pred), 3))
print("evaluate() with weighted:", round(evaluate(weighted, y_true, y_pred), 3))

---
## 9. __eq__, __lt__ and Friends: Comparison and Ordering <a id="9"></a>

Python maps all comparison operators to dunder methods:

| Operator | Method | Meaning |
|---|---|---|
| `==` | `__eq__` | equal |
| `!=` | `__ne__` | not equal (auto-derived from `__eq__`) |
| `<` | `__lt__` | less than |
| `<=` | `__le__` | less than or equal |
| `>` | `__gt__` | greater than |
| `>=` | `__ge__` | greater than or equal |

You rarely need all six. Use `@functools.total_ordering` to derive the rest from just `__eq__` and one comparison.

In [ ]:
from functools import total_ordering

@total_ordering
class ModelResult:
    """Stores the result of a training run. Comparable by accuracy."""

    def __init__(self, name: str, accuracy: float, n_estimators: int = 100):
        self.name = name
        self.accuracy = accuracy
        self.n_estimators = n_estimators

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, ModelResult):
            return NotImplemented
        return self.accuracy == other.accuracy

    def __lt__(self, other: "ModelResult") -> bool:
        if not isinstance(other, ModelResult):
            return NotImplemented
        return self.accuracy < other.accuracy

    def __repr__(self) -> str:
        return f"ModelResult({self.name!r}, accuracy={self.accuracy:.4f})"


results = [
    ModelResult("rf_100",  accuracy=0.872),
    ModelResult("rf_200",  accuracy=0.891),
    ModelResult("rf_50",   accuracy=0.854),
    ModelResult("lr",      accuracy=0.841),
]

print("Best model:  ", max(results))
print("Worst model: ", min(results))
print()
print("Sorted by accuracy:")
for r in sorted(results):
    print(" ", r)
print()
# All comparison operators work via @total_ordering
print("rf_200 > rf_100:", results[1] > results[0])
print("rf_50  < rf_200:", results[2] < results[1])

---
## 10. __hash__: The Consequence of __eq__ <a id="10"></a>

When you define `__eq__`, Python automatically sets `__hash__ = None`, making your object **unhashable**,  it can no longer be used as a dict key or in a set.

This is intentional: if two objects are equal, they must produce the same hash. Python can't guarantee that without knowing your equality logic.

You have three options:

1. **Do nothing**: accept that instances are unhashable (fine for most ML objects)
2. **Define `__hash__`**: hash the fields that define equality
3. **Use `@dataclass(frozen=True)`**: generates both correctly (covered in notebook 07)

In [ ]:
# Demonstrate the problem
class ModelResultUnhashable:
    def __init__(self, name: str, accuracy: float):
        self.name = name
        self.accuracy = accuracy

    def __eq__(self, other):
        if not isinstance(other, ModelResultUnhashable):
            return NotImplemented
        return self.accuracy == other.accuracy

    # __hash__ is now None — unhashable


r = ModelResultUnhashable("rf", 0.87)
try:
    s = {r}   # sets need hashable objects
except TypeError as e:
    print(f"TypeError: {e}")
    print("Defining __eq__ removed __hash__.")
print()

# Fix: define __hash__ based on the same fields used in __eq__
class ModelResultHashable:
    def __init__(self, name: str, accuracy: float):
        self.name = name
        self.accuracy = accuracy

    def __eq__(self, other):
        if not isinstance(other, ModelResultHashable):
            return NotImplemented
        return self.accuracy == other.accuracy

    def __hash__(self) -> int:
        return hash(self.accuracy)  # same field as __eq__


r1 = ModelResultHashable("rf_a", 0.87)
r2 = ModelResultHashable("rf_b", 0.87)  # equal to r1
r3 = ModelResultHashable("lr",   0.83)

result_set = {r1, r2, r3}
print(f"Set of results: {len(result_set)} unique (r1 == r2, so deduplicated)")

cache = {r1: "fitted_model_object"}
print(f"Dict lookup: {cache[r2]!r}")  # r2 == r1, so same key

---
## 11. __add__ and Arithmetic Methods <a id="11"></a>

Arithmetic dunders let your objects work with `+`, `-`, `*`, `/` and their in-place variants `+=`, `-=`, etc.

| Operator | Method | In-place |
|---|---|---|
| `+` | `__add__` | `__iadd__` |
| `-` | `__sub__` | `__isub__` |
| `*` | `__mul__` | `__imul__` |
| `/` | `__truediv__` | `__itruediv__` |

For DS objects the most natural use is `+` to combine datasets or merge metrics.

In [ ]:
class Dataset:
    def __init__(self, X: pd.DataFrame, y: pd.Series, name: str = "dataset"):
        self.X = X
        self.y = y
        self.name = name
        self._splits: dict = {}

    def split(self, test_size: float = 0.2, random_state: int = 42) -> "Dataset":
        X_tr, X_te, y_tr, y_te = train_test_split(
            self.X, self.y, test_size=test_size,
            random_state=random_state, stratify=self.y
        )
        self._splits = {"X_train": X_tr, "X_test": X_te,
                        "y_train": y_tr, "y_test": y_te}
        return self

    @property
    def train(self): return self._splits["X_train"], self._splits["y_train"]
    @property
    def test(self):  return self._splits["X_test"],  self._splits["y_test"]
    @property
    def feature_names(self): return self.X.columns.tolist()

    def __len__(self) -> int: return len(self.X)
    def __getitem__(self, key): return self.X[key] if isinstance(key, str) else self.X.iloc[key]
    def __contains__(self, col: str) -> bool: return col in self.X.columns
    def __bool__(self) -> bool: return len(self) > 0
    def __iter__(self):
        for _, row in self.X.iterrows(): yield row
    def __str__(self) -> str:
        return f"Dataset('{self.name}'): {len(self)} rows x {self.X.shape[1]} features"
    def __repr__(self) -> str:
        return f"Dataset(name={self.name!r}, shape={self.X.shape})"

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Dataset): return NotImplemented
        return self.X.equals(other.X) and self.y.equals(other.y)

    def __add__(self, other: "Dataset") -> "Dataset":
        """Concatenate two datasets vertically: ds1 + ds2"""
        if not isinstance(other, Dataset):
            return NotImplemented
        if self.feature_names != other.feature_names:
            raise ValueError(
                f"Cannot add datasets with different features: "
                f"{self.feature_names} vs {other.feature_names}"
            )
        return Dataset(
            pd.concat([self.X, other.X], ignore_index=True),
            pd.concat([self.y, other.y], ignore_index=True),
            name=f"{self.name}+{other.name}",
        )

    def __iadd__(self, other: "Dataset") -> "Dataset":
        """In-place concatenation: ds1 += ds2"""
        result = self.__add__(other)
        self.X = result.X
        self.y = result.y
        self.name = result.name
        return self


np.random.seed(42)
X1 = pd.DataFrame({"tenure": [12, 5, 24], "charges": [70.0, 30.0, 90.0]})
y1 = pd.Series([1, 0, 1], name="churn")
X2 = pd.DataFrame({"tenure": [3, 8],  "charges": [20.0, 55.0]})
y2 = pd.Series([0, 0], name="churn")

ds1 = Dataset(X1, y1, "batch_1")
ds2 = Dataset(X2, y2, "batch_2")

combined = ds1 + ds2
print(f"ds1 + ds2: {combined}")
print()

# In-place
ds1 += ds2
print(f"ds1 after +=: {ds1}")
print()

# Feature mismatch raises a clear error
X3 = pd.DataFrame({"wrong_col": [1, 2]})
y3 = pd.Series([0, 1], name="churn")
ds3 = Dataset(X3, y3, "bad")
try:
    _ = ds1 + ds3
except ValueError as e:
    print(f"ValueError: {e}")

---
## 12. Exercise: Build a MetricsTracker <a id="12"></a>

Apply what you've learned to build a `MetricsTracker`, an object that collects evaluation metrics across training epochs.

**Requirements:**

1. `__init__` accepts a `name: str`; stores metrics internally as `dict[str, list[float]]`
2. `__len__` returns the number of epochs recorded
3. `__getitem__`: `tracker["accuracy"]` returns the list of values for that metric
4. `__contains__`: `"accuracy" in tracker` checks if a metric was recorded
5. `__bool__`: `True` if at least one epoch has been recorded
6. `__call__`: `tracker(epoch_metrics: dict)` records one epoch's metrics
7. `__add__`: `tracker1 + tracker2` merges two trackers (concatenates their metric lists)
8. `__repr__`: shows name and number of epochs

In [ ]:
class MetricsTracker:
    pass  # TODO


# --- Tests ---
tracker = MetricsTracker("run_001")

# __bool__ before any data
print(f"Empty tracker bool: {bool(tracker)}")

# __call__ to record epochs
tracker({"accuracy": 0.81, "auc": 0.87})
tracker({"accuracy": 0.85, "auc": 0.90})
tracker({"accuracy": 0.88, "auc": 0.92})

print(f"Epochs recorded: {len(tracker)}")          # __len__
print(f"Accuracy values: {tracker['accuracy']}")   # __getitem__
print(f"'auc' in tracker: {'auc' in tracker}")     # __contains__
print(f"Bool after data:  {bool(tracker)}")        # __bool__
print(repr(tracker))                                # __repr__

# __add__
tracker2 = MetricsTracker("run_002")
tracker2({"accuracy": 0.90, "auc": 0.94})

combined = tracker + tracker2
print(f"Combined accuracy: {combined['accuracy']}")

Do you need a bit of help? Here you can find a possible implementation of the class.

> <details>
> <summary>Click here</summary>
> 
> ```python
> class MetricsTracker:
>     def __init__(self, name: str):
>         self.name = name
>         self._data: dict[str, list[float]] = {}
>
>     def __len__(self) -> int:
>         if not self._data:
>             return 0
>         return len(next(iter(self._data.values())))
>
>     def __getitem__(self, metric: str) -> list[float]:
>         return self._data[metric]
>
>     def __contains__(self, metric: str) -> bool:
>         return metric in self._data
>
>     def __bool__(self) -> bool:
>         return len(self) > 0
>
>     def __call__(self, epoch_metrics: dict[str, float]) -> None:
>         for key, value in epoch_metrics.items():
>             self._data.setdefault(key, []).append(value)
>
>     def __add__(self, other: "MetricsTracker") -> "MetricsTracker":
>         result = MetricsTracker(f"{self.name}+{other.name}")
>         all_keys = set(self._data) | set(other._data)
>         for key in all_keys:
>             result._data[key] = (
>                 self._data.get(key, []) + other._data.get(key, [])
>             )
>         return result
>
>     def __repr__(self) -> str:
>         return f"MetricsTracker({self.name!r}, epochs={len(self)}, metrics={list(self._data)})"
> ```
> </details>